# Early 17c formal-rejection search

Find a resolution (~1600–1620) where a request was sent back for not meeting formal requirements.

**Do not invent 17th-century wording.** Fill `SEARCH_TERMS` with **Python regex** patterns you have attested (notes, republic_ner / phrase inventories, editions). Empty list → no search.

Patterns are **AND**ed (`re.IGNORECASE`). Full-corpus regex AND is Approach 2 (`resolutions_flat`); ES is a year-window preview with the same Python patterns client-side.


In [ ]:
from __future__ import annotations

import re

import pandas as pd
from data_io import load
from formulation.republic_es import RepublicEsClient

client = RepublicEsClient()

# Attested Python regex only — edit this list; do not add guessed synonyms.
SEARCH_TERMS: list[str] = [
    # r"…",
]

YEAR_MIN, YEAR_MAX = 1600, 1625

PATTERNS = [re.compile(p, re.IGNORECASE) for p in SEARCH_TERMS]
if not PATTERNS:
    print(
        "SEARCH_TERMS is empty — fill attested Python regex, then re-run this cell "
        "before Approach 1/2 (do not invent 17c terminology)."
    )


def and_contains(text: pd.Series, patterns: list[re.Pattern[str]]) -> pd.Series:
    """True where every regex matches (case-insensitive)."""
    if not patterns:
        raise ValueError("PATTERNS is empty — set SEARCH_TERMS in the setup cell.")
    term_hits = pd.concat(
        {
            p.pattern: text.str.contains(p.pattern, case=False, na=False, regex=True)
            for p in patterns
        },
        axis=1,
    )
    return term_hits.all(axis=1)


def and_contains_str(text: str, patterns: list[re.Pattern[str]]) -> bool:
    if not patterns:
        raise ValueError("PATTERNS is empty — set SEARCH_TERMS in the setup cell.")
    return all(p.search(text) is not None for p in patterns)


## Approach order

Run the **setup cell** (imports + `SEARCH_TERMS`) first, then Approach 1 and/or 2.


In [ ]:
# helpers live in the setup cell (and_contains / and_contains_str)


## Approach 1: ES year window + same Python regex AND (client-side)


In [ ]:
index = client.config.resolutions_index
body = {
    "size": 500,
    "query": {
        "range": {"metadata.session_year": {"gte": YEAR_MIN, "lte": YEAR_MAX}}
    },
    "_source": [
        "id",
        "metadata.session_year",
        "metadata.session_date",
        "metadata.proposition_type",
        "paragraphs.text",
    ],
}

print(
    f"ES year window {YEAR_MIN}–{YEAR_MAX} on {index}; "
    f"AND {len(PATTERNS)} Python regex client-side..."
)
response = client.search(index, body)
raw_hits = response.get("hits", {}).get("hits") or []
total = response.get("hits", {}).get("total")

rows = []
for hit in raw_hits:
    src = hit.get("_source") or {}
    text = " ".join(
        p.get("text") or ""
        for p in (src.get("paragraphs") or [])
        if isinstance(p, dict)
    )
    if not and_contains_str(text, PATTERNS):
        continue
    meta = src.get("metadata") or {}
    rows.append(
        {
            "id": hit.get("_id"),
            "year": meta.get("session_year"),
            "proposition_type": meta.get("proposition_type"),
            "snippet": text[:200],
        }
    )

print(f"Scanned {len(raw_hits)} ES hits (total≈{total}); regex AND matches: {len(rows)}")
print("(Full-corpus regex AND → Approach 2.)\n")
preview = pd.DataFrame(rows[:20])
preview


## Approach 2: `resolutions_flat` (same Python regex AND)


In [ ]:
flat = load("resolutions_flat")
early = flat.loc[flat["year"].between(YEAR_MIN, YEAR_MAX)].copy()
print(f"Early slice {YEAR_MIN}–{YEAR_MAX}: {len(early)} rows")

mask = and_contains(early["resolutions_text"].astype(str), PATTERNS)
hits = early.loc[mask, ["id", "date", "resolutions_text"]].assign(
    snippet=lambda d: d["resolutions_text"].astype(str).str.slice(0, 200)
)
print(f"regex AND-match rows: {len(hits)}")
hits[["id", "date", "snippet"]]


## Inspect one candidate (paste an id from above)


In [ ]:
candidate_id = ""  # paste attested hit id

if not candidate_id:
    print("Set candidate_id to an id from the hit table.")
else:
    row = flat.loc[flat["id"].eq(candidate_id), ["id", "date", "resolutions_text"]]
    if row.empty:
        print(f"ID {candidate_id!r} not found")
    else:
        display(row)
        print(row["resolutions_text"].iloc[0])
